In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states, save_breakthrough_states, load_forgetting_states, save_forgetting_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores, make_forgettable_subset
from loka.geometrical_methods import herding_order

import os
import pandas as pd
from dotenv import load_dotenv
import random

load_dotenv()
BASE_DIR = os.getenv("BASE_DIR")

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

X,Y = load_breakthrough_states(f"{BASE_DIR}/data/game_states/o.txt")

In [ ]:
order = herding_order(X, 0.11, device)

In [ ]:
epochs = 200

pbar = tqdm(
    total=epochs,
    desc="training",
    bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]",
)

X_t, Y_t = X, Y

torch.manual_seed(SEED)
model = BreakthroughCNN()
forgetter = ForgettingScores(model, seed=SEED)

forgetter.train(
    X = X_t,
    Y = Y_t,
    epochs = epochs,
    batch_size= 512,
    random_permuations=True,
    device=device,
    pbar=pbar,
    pbar_increment=1
)

C = forgetter.getForgettingCounts()
C

pbar.close()

In [ ]:
def train(model, optimizer, loss_fn, X, Y, epochs, pbar, pbar_inc, n_points):
    X_t = X[:n_points].to(device)
    Y_t = Y[:n_points].to(device)

    for _ in range(epochs):
        perm = torch.randperm(len(X_t), device=device)
        
        for i in range(0, len(X_t), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            xb, yb = X_t[idx], Y_t[idx]

            optimizer.zero_grad()
            batch_loss = loss_fn(model(xb), yb)
            batch_loss.backward()
            optimizer.step()
        
        pbar.update(pbar_inc)

In [ ]:
TOTAL_EPOCHS = 50
BATCH_SIZE = 512

start, end, steps = 0.01, 0.1, 50
splits = np.linspace(start, end, num=steps, endpoint=True)

ratios = [0.01, 0.05, 0.1, 0.2, 0.3]

data_dir = f"{BASE_DIR}/data/forgetting_counts/herding_forget_mix"
os.makedirs(data_dir, exist_ok=True)  # create the folder if it doesn't exist yet

n_points_per_split = [int(s * len(X)) for s in splits]
total_work = len(ratios) * TOTAL_EPOCHS * sum(n_points_per_split)

all_mse_results = list()
all_prc_results = list()

pbar = tqdm(total=total_work, desc='Training',
            bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]")

X_t = torch.FloatTensor(X).to(device)
Y_t = torch.FloatTensor(Y).to(device)

for ratio in ratios:
    mse_values = list()
    prc_values = list()

    for s, pbar_inc in zip(splits, n_points_per_split):
        n_points = int(s * len(X))

        S_size = int(s * len(X))
        forgettable_size = int(s * len(X) * ratio)

        X_S, Y_S = make_forgettable_subset(X_t, Y_t, C, order, S_size, forgettable_size)

        torch.manual_seed(SEED)
        model = BreakthroughCNN().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.MSELoss()

        train(model, optimizer, loss_fn, X_S, Y_S, TOTAL_EPOCHS, pbar, pbar_inc, n_points)

        model.eval()
        with torch.no_grad():
            sq_err = 0.0
            for k in range(0, len(X_t), BATCH_SIZE):
                xb, yb = X_t[k:k + BATCH_SIZE], Y_t[k:k + BATCH_SIZE]
                sq_err += ((model(xb) - yb) ** 2).sum().item()
            mse = sq_err / Y_t.numel()
        mse_values.append(mse)
        prc_values.append(s)

        del model, optimizer
        torch.cuda.empty_cache()

    all_mse_results.append(mse_values)
    all_prc_results.append(prc_values)

    # Save this ratio's results as soon as it finishes
    h = int(round(ratio * 100))
    csv_file = os.path.join(data_dir, f"herding_forget_mix_{100 - h}_{h}.csv")
    pd.DataFrame({'split': prc_values, 'mse': mse_values}).to_csv(csv_file, index=False)

pbar.close()